# RAISE-26 News Analysis: AI, Cognition, and Human Agency

In this notebook, we analyze the RAISE-26 news dataset using Python and modern NLP tools.
Our goal is to understand how artificial intelligence and its impact on human behavior,
cognition, and agency are represented in news media.

We combine data cleaning, exploratory analysis, semantic modeling, and large language
model synthesis to extract both quantitative patterns and qualitative insights.


## Environment Setup

We use Python-based data science and NLP libraries to support this analysis.

- pandas, numpy: structured data handling  
- matplotlib, seaborn: visualization  
- transformers, sentence-transformers: language models and embeddings  
- scikit-learn: clustering and analysis  

All libraries used are publicly available and do not require authentication.


In [ ]:
import os
print("Current directory:", os.getcwd())
print("Files here:", os.listdir())

# Install required packages (no duplicates)
!pip install -q -U "pandas==2.2.2" matplotlib seaborn
!pip install -q -U transformers accelerate sentence-transformers scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline

sns.set_theme(style="whitegrid")

## Loading and Preparing the Data

To keep the analysis clean and reproducible, we define a helper function to load and
standardize the RAISE-26 dataset. This step handles date formatting, boolean fields,
text normalization, and multi-label class parsing.

Since each article may belong to multiple AI impact categories, class labels are
stored as lists and later expanded for analysis.


In [ ]:
def load_and_clean_raise26(file_path):
    df = pd.read_csv(file_path)

    # Convert date column
    df['date'] = pd.to_datetime(df['date'])

    # Fix boolean formatting
    if df['is_weekend'].dtype == 'object':
        df['is_weekend'] = df['is_weekend'].str.strip().str.upper() == 'TRUE'

    # Normalize quotation marks for NLP
    df['title'] = df['title'].str.replace('‘', "'").str.replace('’', "'")

    # Parse multi-label classes
    df['class_list'] = df['classes_str'].fillna('').str.split(';')
    df['class_list'] = df['class_list'].apply(
        lambda x: [c.strip() for c in x if c.strip()]
    )

    return df

df = load_and_clean_raise26('dataset_A_news_full_10500.csv')

# Add year for time-based analysis
df['year'] = df['date'].dt.year

# Explode multi-label classes
df_exploded = df.explode('class_list')
df_exploded['year'] = df_exploded['date'].dt.year

print(f"Dataset synced with {len(df)} rows.")
print(f"Exploded dataset contains {len(df_exploded)} class-level entries.")


## First Look at the Data

Before applying modeling techniques, we explore the dataset to understand its
structure and overall content. This helps ensure that later analysis is grounded
in the actual characteristics of the data.


In [ ]:
# Quick overview of the dataset structure
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"Unique sources: {df['source'].nunique()}")
print()

# Preview the first few rows
df.head()


In [ ]:
# Summary statistics for numeric columns
df.describe()


In [ ]:
# Top 10 news sources by article count
print("Top 10 Sources:")
print(df['source'].value_counts().head(10))
print()

# Most common AI impact categories
print("Top 10 AI Impact Categories:")
print(df_exploded['class_list'].value_counts().head(10))


## How AI Themes Change Over Time

To understand how AI-related topics evolve, we examine the most common AI impact
categories and track how frequently they appear across different years.

This allows us to identify persistent themes as well as shifts in attention over time.


In [ ]:
plt.figure(figsize=(12, 6))

top_classes = df_exploded['class_list'].value_counts().head(10).index
df_time = df_exploded[df_exploded['class_list'].isin(top_classes)]

sns.histplot(
    data=df_time,
    x='year',
    hue='class_list',
    multiple='stack',
    kde=False
)

plt.title('Evolution of AI Impact Themes Over Time')
plt.ylabel('Number of Headlines')
plt.xlabel('Year')
plt.show()


Some AI themes appear consistently over time, while others spike during certain years.
These changes likely reflect major technological developments or public debates that
temporarily increase attention around specific AI-related issues.


## When Is AI News Published?

We compare AI-related news volume between weekdays and weekends to better understand
how AI discourse fits into traditional news production cycles.


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='is_weekend')
plt.title('AI News Volume: Weekdays vs. Weekends')
plt.xlabel('Is Weekend')
plt.ylabel('Article Count')
plt.show()


AI-related articles appear more frequently on weekdays, suggesting that AI discourse
is closely tied to professional, academic, and policy-driven news cycles rather than
casual or entertainment-focused media.


## Understanding Cognitive AI Through Semantics

To move beyond surface-level keyword analysis, we focus on headlines related to
cognitive aspects of AI and apply semantic embeddings.

This approach allows us to group headlines based on meaning rather than exact wording.


In [ ]:
import os
import warnings
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from transformers import pipeline, logging as transformers_logging

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
warnings.filterwarnings("ignore")
transformers_logging.set_verbosity_error()

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

cognitive_df = df[df['classes_str'].str.contains('Cognitive', na=False)].copy()

if not cognitive_df.empty:
    headlines = cognitive_df['title'].tolist()
    embeddings = embed_model.encode(headlines)

    kmeans = KMeans(n_clusters=5, n_init='auto', random_state=42)
    cognitive_df['cluster'] = kmeans.fit_predict(embeddings)
    print(f" Assigned {len(cognitive_df['cluster'].unique())} clusters to {len(cognitive_df)} headlines")

    print("\n### THEMATIC CLUSTERING RESULTS ###\n")

    for cluster_id in range(5):
        samples = cognitive_df[cognitive_df['cluster'] == cluster_id]['title'].head(3).tolist()
        sentiments = sentiment_analyzer(samples)

        print(f"Cluster {cluster_id}:")
        for i, text in enumerate(samples):
            print(f" - [{sentiments[i]['label']}] {text}")
        print("-" * 40)
else:
    print("No cognitive headlines found.")

In [ ]:
# Visualize the semantic clusters in 2D using PCA
from sklearn.decomposition import PCA

if not cognitive_df.empty:
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(embeddings)

    cognitive_df['x'] = coords[:, 0]
    cognitive_df['y'] = coords[:, 1]

    plt.figure(figsize=(10, 7))
    sns.scatterplot(
        data=cognitive_df,
        x='x', y='y',
        hue='cluster',
        palette='Set2',
        alpha=0.7
    )
    plt.title('Semantic Clusters of Cognitive AI Headlines (PCA)')
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.legend(title='Cluster')
    plt.show()


Each cluster reflects a different narrative around AI and human cognition, such as
decision-making support, creativity, or concerns about automation.

Sentiment analysis adds context by showing whether these narratives are framed
optimistically or with caution in the media.


## AI, Human Agency, and High-Level Patterns

To complement the clustering analysis, we use a large language model to classify
how AI's impact on human agency is framed in each headline.

Unlike the previous Cell which grouped headlines by topic (semantic clustering), this analysis
makes a binary judgment: Does the headline portray AI as ENHANCING human agency
(giving humans more control/capability) or CHALLENGING human agency (reducing
human autonomy/control)?

This allows us to measure the overall stance of media discourse on AI and human agency.

In [ ]:
# FLAN-T5 Classification of Cognitive Headlines

from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

print("Loading FLAN-T5 model for classification...")
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base").to(device)
model.eval()
print("Model loaded!")

# Filter cognitive headlines (same criteria as Cell 10)
cognitive_df = df[df['classes_str'].str.contains('Cognitive', na=False)].copy()
headlines = cognitive_df['title'].tolist()
total = len(headlines)

print(f"\nTotal cognitive headlines to analyze: {total}")

print("\n" + "="*70)
print("CLASSIFYING AI'S IMPACT ON HUMAN AGENCY")
print("="*70)
print("Binary classification: ENHANCES vs CHALLENGES (no neutral)")
print(f"Processing {total} headlines...")
print("="*70)

results = {"Enhances": 0, "Challenges": 0}

batch_size = 8
progress_bar = tqdm(total=total, desc="Processing")

for i in range(0, len(headlines), batch_size):
    batch = headlines[i:i+batch_size]

    for headline in batch:
        try:
            prompt = f"Does AI enhance or challenge human agency? Answer enhance or challenge: {headline}"

            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)

            result = tokenizer.decode(outputs[0], skip_special_tokens=True).lower()

            if "enhance" in result:
                results["Enhances"] += 1
            else:
                results["Challenges"] += 1

        except Exception:
            results["Challenges"] += 1

        progress_bar.update(1)

progress_bar.close()

print("\n" + "="*70)
print("AGENCY CLASSIFICATION RESULTS")
print("="*70)
print(f"Total headlines analyzed: {total}")
print(f"Enhances human agency: {results['Enhances']} ({results['Enhances']/total*100:.1f}%)")
print(f"Challenges human agency: {results['Challenges']} ({results['Challenges']/total*100:.1f}%)")

# Visualization
plt.figure(figsize=(6, 5))
bars = plt.bar(['Enhances', 'Challenges'], [results['Enhances'], results['Challenges']],
               color=['#2ecc71', '#e74c3c'])
plt.title(f'AI Agency Classification\n(n={total} cognitive headlines)', fontsize=12)
plt.ylabel('Number of Headlines')
plt.ylim(0, total)
for bar, v in zip(bars, [results['Enhances'], results['Challenges']]):
    plt.text(bar.get_x() + bar.get_width()/2, v + total*0.02, str(v),
             ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# Cleanup
del model
del tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:

import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

from transformers import pipeline
print("Loading zero-shot classification model (DistilBART)...")
classifier = pipeline("zero-shot-classification",
                      model="typeform/distilbert-base-uncased-mnli",
                      device=0 if torch.cuda.is_available() else -1)
print("Model loaded!")

def classify_headline(headline):
    candidate_labels = ["enhances human agency", "challenges human agency"]
    output = classifier(headline, candidate_labels)
    return output['labels'][0].split()[0]

# Use the same cognitive headlines as Cell 10
cognitive_df = df[df['classes_str'].str.contains('Cognitive', na=False)].copy()
headlines = cognitive_df['title'].tolist()
total = len(headlines)

print(f"\nTotal cognitive headlines to analyze: {total}")

print("\n" + "="*70)
print("CLASSIFYING AI'S IMPACT ON HUMAN AGENCY")
print("="*70)
print(f"Processing {total} headlines...")
print("="*70)

results = {"enhances": 0, "challenges": 0}

for i, headline in enumerate(tqdm(headlines, desc="Processing")):
    try:
        result = classify_headline(headline)
        if "enhance" in result.lower():
            results["enhances"] += 1
        else:
            results["challenges"] += 1
    except Exception as e:
        results["challenges"] += 1

print("\n" + "="*70)
print("AGENCY CLASSIFICATION RESULTS")
print("="*70)
print(f"Total headlines analyzed: {total}")
print(f"Enhances human agency: {results['enhances']} ({results['enhances']/total*100:.1f}%)")
print(f"Challenges human agency: {results['challenges']} ({results['challenges']/total*100:.1f}%)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Agency classification results
axes[0].bar(['Enhances', 'Challenges'], [results['enhances'], results['challenges']],
            color=['#2ecc71', '#e74c3c'])
axes[0].set_title(f'Agency Classification\n(n={total} cognitive headlines)', fontsize=12)
axes[0].set_ylabel('Number of Headlines')
axes[0].set_ylim(0, total)
for i, v in enumerate([results['enhances'], results['challenges']]):
    axes[0].text(i, v + total*0.02, str(v), ha='center', fontweight='bold')

# Plot 2: Percentage
percentages = [results['enhances']/total*100, results['challenges']/total*100]
axes[1].bar(['Enhances', 'Challenges'], percentages, color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Agency Classification (Percentage)', fontsize=12)
axes[1].set_ylabel('Percentage (%)')
axes[1].set_ylim(0, 100)
for i, v in enumerate(percentages):
    axes[1].text(i, v + 2, f"{v:.1f}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("INTERPRETATION")
print("="*70)

enhance_pct = results['enhances']/total*100
challenge_pct = results['challenges']/total*100

if enhance_pct > challenge_pct + 10:
    print(f"\n{enhance_pct:.0f}% of cognitive headlines frame AI as ENHANCING human agency.")
    print("This suggests media predominantly portrays AI as a tool that")
    print("augments human decision-making and control.")
elif challenge_pct > enhance_pct + 10:
    print(f"\n{challenge_pct:.0f}% of cognitive headlines frame AI as CHALLENGING human agency.")
    print("This suggests media emphasizes AI's potential to reduce")
    print("human autonomy and control.")
else:
    print(f"\nResults are nearly balanced: {enhance_pct:.0f}% Enhances vs {challenge_pct:.0f}% Challenges.")
    print("Media discourse on AI and human agency is mixed.")

The language model highlights recurring ideas around shifting human agency, including
greater reliance on AI systems, augmented decision-making, and ongoing questions
about autonomy and control.

These quantitative classifications provide a systematic way to understand how media discourse frames AI's relationship to human decision-making and autonomy.

## Conclusion

This notebook presents a multi-layered analysis of the RAISE-26 news dataset by
combining exploratory data analysis, semantic clustering, sentiment modeling,
and large language model synthesis.

Together, these methods provide insight into how AI’s impact on human behavior,
cognition, and agency is represented in contemporary news media.
